# Transformer Baseline: Resume to 100k + Evaluation

Resume the Europarl transformer from the saved checkpoint (`out/transformer_full/ckpt.pt`) and train to **100,000 iterations**, then run standardized evals.

**Architecture (from checkpoint):** 4 layers, 4 heads, 256 embd, block size 128, ~131M params  
**Config:** `train_europarl_transformer_baseline_16senses`  
**Paused at:** ~88,000 iterations (Dec 2025 run + local resume)

### Colab only (skip if running locally)

Run the cell below if you haven't cloned the repo yet.

In [ ]:
# Colab: clone repo (run once per session)
import subprocess
import sys
from pathlib import Path

if not Path('train.py').exists():
    subprocess.run(['git', 'clone', 'https://github.com/kavyavenk/multilingual-backpacks.git'], check=True)
    import os
    os.chdir('multilingual-backpacks')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers', 'datasets', 'scipy', 'tqdm', 'numpy',
                'torch', 'matplotlib', 'pandas'], check=True)
print('Ready:', Path.cwd())

## 1. Setup

In [ ]:
import os
import sys
import json
import re
import subprocess
from pathlib import Path

import torch
import matplotlib.pyplot as plt
import pandas as pd


def find_repo_root():
    """Locate multilingual-backpacks root (works in Colab, nested clones, local)."""
    def is_root(p: Path) -> bool:
        return (p / 'train.py').exists() and (p / 'config').is_dir()

    # Walk up from current working directory
    for p in [Path.cwd(), *Path.cwd().parents]:
        if is_root(p):
            return p.resolve()

    # Common Colab / local paths
    extra = [
        Path('/content/multilingual-backpacks'),
        Path('/content/multilingual-backpacks/multilingual-backpacks'),
        Path.home() / 'Documents' / 'multilingual-backpacks',
        Path('/Users/itskavya/Documents/multilingual-backpacks'),
    ]
    for p in extra:
        if is_root(p):
            return p.resolve()
        # nested clone (Colab sometimes double-clones)
        for child in p.glob('**/train.py'):
            candidate = child.parent
            if is_root(candidate):
                return candidate.resolve()

    return None


ROOT = find_repo_root()
if ROOT is None:
    raise FileNotFoundError(
        'Could not find repo root. Run this first:\n'
        '  !git clone https://github.com/kavyavenk/multilingual-backpacks.git\n'
        '  %cd multilingual-backpacks'
    )

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

OUT_DIR = ROOT / 'out' / 'transformer_full'
CKPT_PATH = OUT_DIR / 'ckpt.pt'
CONFIG_PATH = ROOT / 'config' / 'train_europarl_transformer_baseline_16senses.py'
DATA_DIR = 'europarl'  # relative to repo; train.py resolves this
TARGET_ITERS = 100_000

# Auto-detect device
if torch.cuda.is_available():
    DEVICE = 'cuda'
elif torch.backends.mps.is_available():
    DEVICE = 'mps'
else:
    DEVICE = 'cpu'

print(f'Root: {ROOT}')
print(f'Device: {DEVICE}')
print(f'Checkpoint: {CKPT_PATH}')

## 1b. Load checkpoint + data (Colab)

The repo clone does **not** include `ckpt.pt` (~1.5 GB). Pick **one** option below.

| Option | When to use |
|--------|-------------|
| **A. Google Drive** | You saved checkpoints to Drive from a prior Colab run |
| **B. Manual upload** | You have `ckpt.pt` on your Mac |
| **C. Skip** | Checkpoint already at `out/transformer_full/ckpt.pt` |

In [ ]:
import shutil
import os
import sys
from pathlib import Path

# Re-sync paths after setup
ROOT = Path.cwd()
if not (ROOT / 'train.py').exists():
    ROOT = Path('/content/multilingual-backpacks')  # fallback
OUT_DIR = ROOT / 'out' / 'transformer_full'
CKPT_PATH = OUT_DIR / 'ckpt.pt'
OUT_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_SOURCE = 'drive'  # 'drive' | 'upload' | 'skip'

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if CKPT_PATH.exists():
    print(f'✓ Checkpoint already present: {CKPT_PATH}')
    print(f'  Size: {CKPT_PATH.stat().st_size / 1e9:.2f} GB')
elif CHECKPOINT_SOURCE == 'drive' and IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    drive_candidates = [
        '/content/drive/MyDrive/multilingual-backpacks-checkpoints/transformer_full',
        '/content/drive/MyDrive/transformer_full',
        '/content/drive/MyDrive/multilingual-backpacks/out/transformer_full',
    ]
    restored = False
    for drive_dir in drive_candidates:
        src = Path(drive_dir) / 'ckpt.pt'
        if src.exists():
            shutil.copy2(src, CKPT_PATH)
            log_src = Path(drive_dir) / 'training_log.json'
            if log_src.exists():
                shutil.copy2(log_src, OUT_DIR / 'training_log.json')
            print(f'✓ Copied checkpoint from {drive_dir}')
            restored = True
            break
    if not restored:
        print('❌ No checkpoint on Drive. Set CHECKPOINT_SOURCE = "upload" and re-run.')
elif CHECKPOINT_SOURCE == 'upload' and IN_COLAB:
    from google.colab import files
    print('Select ckpt.pt from your computer (~1.5 GB)...')
    uploaded = files.upload()
    for name, data in uploaded.items():
        dest = CKPT_PATH if name.endswith('ckpt.pt') else OUT_DIR / name
        with open(dest, 'wb') as f:
            f.write(data)
        print(f'✓ Saved {dest}')
elif CHECKPOINT_SOURCE == 'skip':
    pass
else:
    print(f'Local mode: place ckpt at {CKPT_PATH}')

if not CKPT_PATH.exists():
    raise FileNotFoundError(
        f'Missing {CKPT_PATH}\n\n'
        'On Colab: upload out/transformer_full/ckpt.pt from your Mac (~1.5 GB).\n'
        '  Option A: set CHECKPOINT_SOURCE="upload" and re-run this cell\n'
        '  Option B: copy to Drive at MyDrive/multilingual-backpacks-checkpoints/transformer_full/ckpt.pt\n'
        'On Mac: training is already at out/transformer_full/ckpt.pt locally'
    )

In [ ]:
# Europarl tokenized data (required for training)
import subprocess
import sys

data_dir = ROOT / 'data' / 'europarl'
if not (data_dir / 'train.bin').exists():
    print('Preparing Europarl data (first time only, ~10 min)...')
    subprocess.run([sys.executable, 'prepare_europarl.py', '--out_dir', 'data/europarl'],
                   cwd=ROOT, check=True)
else:
    print(f'✓ Data ready: {data_dir}')

## 2. Inspect current checkpoint

In [ ]:
assert CKPT_PATH.exists(), f'Missing checkpoint: {CKPT_PATH}'

ckpt = torch.load(CKPT_PATH, map_location='cpu', weights_only=False)
cfg = ckpt['config']

n_params = sum(v.numel() for v in ckpt['model'].values())

summary = {
    'iter_num': ckpt.get('iter_num'),
    'best_val_loss': round(ckpt.get('best_val_loss', 0), 4),
    'n_layer': cfg.n_layer,
    'n_head': cfg.n_head,
    'n_embd': cfg.n_embd,
    'block_size': cfg.block_size,
    'max_iters_in_ckpt_config': cfg.max_iters,
    'params_M': round(n_params / 1e6, 2),
}

remaining = max(0, TARGET_ITERS - summary['iter_num'])
summary['remaining_to_100k'] = remaining

pd.Series(summary)

In [ ]:
# Plot existing training curve (if log exists)
log_path = OUT_DIR / 'training_log.json'
if log_path.exists():
    with open(log_path) as f:
        log = json.load(f)
    iters = log['iterations']
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(iters, log['train_loss'], label='train')
    ax[0].plot(iters, log['val_loss'], label='val')
    ax[0].axvline(TARGET_ITERS, color='gray', ls='--', label='100k target')
    ax[0].set_xlabel('iteration')
    ax[0].set_ylabel('loss')
    ax[0].legend()
    ax[0].set_title('Transformer training (so far)')
    if 'learning_rate' in log and log['learning_rate']:
        ax[1].plot(iters, log['learning_rate'])
        ax[1].set_xlabel('iteration')
        ax[1].set_ylabel('learning rate')
        ax[1].set_title('LR schedule')
    plt.tight_layout()
    plt.show()
else:
    print('No training_log.json yet')

## 3. Baseline eval @ ~88k (before resuming)

These are the numbers from the last full eval on the 88k checkpoint (`out/transformer_eval_updated.json`). Re-run the cell below if you want fresh numbers.

In [ ]:
baseline_path = ROOT / 'out' / 'transformer_eval_updated.json'
if baseline_path.exists():
    with open(baseline_path) as f:
        baseline_88k = json.load(f)
    pd.DataFrame([{
        'checkpoint': '~88k',
        'MultiSimLex EN ρ': round(baseline_88k['multisimlex_en']['correlation'], 4),
        'MultiSimLex FR ρ': round(baseline_88k['multisimlex_fr']['correlation'], 4),
        'Perplexity': round(baseline_88k['perplexity']['perplexity'], 2),
        'Emb cosine': round(baseline_88k['embedding_stats']['mean_cosine'], 4),
    }])
else:
    print('No baseline file — run eval cell below')

## 4. Resume training to 100k

Sets `max_iters=100000` in the config, then resumes from `ckpt.pt`.  
**~12k iterations remaining** from 88k → expect ~1–3 hours on MPS/GPU depending on hardware.

> Stop anytime with the notebook interrupt button (■). Checkpoints save every 500 iters on val eval, and on best val loss.

In [ ]:
# Patch max_iters in config file (train.py reads this, not ckpt config)
text = CONFIG_PATH.read_text()
new_text, n = re.subn(r'max_iters=\d+', f'max_iters={TARGET_ITERS}', text, count=1)
assert n == 1, 'Could not patch max_iters in config'
CONFIG_PATH.write_text(new_text)
print(f'Patched {CONFIG_PATH.name}: max_iters={TARGET_ITERS}')

In [ ]:
cmd = [
    sys.executable, 'train.py',
    '--model_type', 'transformer',
    '--config', 'train_europarl_transformer_baseline_16senses',
    '--out_dir', str(OUT_DIR.relative_to(ROOT)),
    '--data_dir', DATA_DIR,
    '--init_from', 'resume',
    '--device', DEVICE,
]
print('Running:', ' '.join(cmd))
print('---')
subprocess.run(cmd, cwd=ROOT, check=True)

In [ ]:
# Verify final checkpoint iteration
ckpt = torch.load(CKPT_PATH, map_location='cpu', weights_only=False)
print(f'Final iter_num: {ckpt["iter_num"]}')
print(f'Best val loss: {ckpt["best_val_loss"]:.4f}')

## 5. Post-training eval (100k checkpoint)

In [ ]:
from run_ckpt_eval import eval_model

results_100k = eval_model(
    name='transformer',
    path=str(OUT_DIR.relative_to(ROOT)),
    device=DEVICE,
    data_dir=str(ROOT / 'data' / 'europarl'),
    max_multisimlex=None,  # full 1888 pairs
)

out_json = ROOT / 'out' / 'transformer_eval_100k.json'
with open(out_json, 'w') as f:
    json.dump(results_100k, f, indent=2)
print(f'Saved: {out_json}')

In [ ]:
def row_from_eval(label, r):
    return {
        'checkpoint': label,
        'iterations': r.get('iterations', ''),
        'MultiSimLex EN ρ': round(r['multisimlex_en']['correlation'], 4) if r.get('multisimlex_en') else None,
        'MultiSimLex FR ρ': round(r['multisimlex_fr']['correlation'], 4) if r.get('multisimlex_fr') else None,
        'Perplexity': round(r['perplexity']['perplexity'], 2) if r.get('perplexity') else None,
        'Emb cosine': round(r['embedding_stats']['mean_cosine'], 4) if r.get('embedding_stats') else None,
    }

rows = []
if baseline_path.exists():
    with open(baseline_path) as f:
        rows.append(row_from_eval('~88k (baseline)', json.load(f)))
rows.append(row_from_eval('100k', results_100k))

pd.DataFrame(rows)

## 6. Optional: compare with backpack

Run full side-by-side eval for both checkpoints (saves `out/ckpt_eval_results.json`).

In [ ]:
cmd = [
    sys.executable, 'run_ckpt_eval.py',
    '--device', DEVICE,
    '--out', 'out/ckpt_eval_results.json',
]
subprocess.run(cmd, cwd=ROOT, check=True)

with open(ROOT / 'out' / 'ckpt_eval_results.json') as f:
    both = json.load(f)

table = []
for name, r in both.items():
    table.append({
        'model': name,
        'EN ρ': round(r['multisimlex_en']['correlation'], 4),
        'FR ρ': round(r['multisimlex_fr']['correlation'], 4),
        'PPL': round(r['perplexity']['perplexity'], 2),
        'emb_cos': round(r['embedding_stats']['mean_cosine'], 4),
    })
pd.DataFrame(table)